**Imports**

In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
from datasets import DatasetDict,Dataset, load_dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

**Load** **Data**

In [ ]:
dataset= load_dataset("syedkhalid0/Sentiment-Analysis")

**Load Pre-trained Model**

In [ ]:
model_path="bert-base-uncased"

In [ ]:
#load model tokenizer
tokenizer= AutoTokenizer.from_pretrained(model_path)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
#load model with binary classification head

model=AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=3
    )

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Set Trainable Parameters**

In [ ]:
#freeze all base model parameters
for name,param in model.base_model.named_parameters():
  param.requires_grad=False

#unfreeze base model pooling layers
for name,param in model.base_model.named_parameters():
  if "pooler" in name:
    param.requires_grad=True

**Data Pre-processing**

In [ ]:
#define text preprocessing
def preprocess_function(example):
  return tokenizer(example["text"],truncation=True)

#preprocess all datasets
tokenized_data= dataset.map(preprocess_function,batched=True)

Map:   0%|          | 0/83989 [00:00<?, ? examples/s]

In [ ]:

#create data collator
data_collator= DataCollatorWithPadding(tokenizer=tokenizer)

**Define Evaluation Metrix**

In [ ]:
#load metrics
accuracy=evaluate.load("accuracy")
auc_score=evaluate.load("roc_auc")

def compute_metrics(eval_pred):
  predictions,labels=eval_pred

  #apply softmax to get probabilities
  probabilities= np.exp(predictions)/np.exp(predictions).sum(-1,keepdims=True)

  #use probabilities of the positive class for ROC AUC
  positive_class_probs=probabilities[:,1]

  #compute auc
  auc=np.round(auc_score.compute(prediction_scores=positive_class_probs,references=labels)['roc_auc'],3)

  #predict most probable class
  predicted_classes=np.argmax(predictions,axis=1)

  #compute accuracy
  accuracy_score=np.round(accuracy.compute(predictions=predicted_classes,references=labels)['accuracy'],3)

  return {"accuracy":accuracy_score,"auc":auc}


**Training Parameters**

In [ ]:
lr=2e-5
batch_size=16
num_epochs=3

training_args=TrainingArguments(
    output_dir="results",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs

    #logging_strategy="epoch",
    #eval_strategy="epoch",
    #save_strategy="epoch",
    #load_best_model_at_end=True

)

**Fine-tune Model**

In [ ]:
trainer= Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.701488
1000,0.658512
1500,0.632011
2000,0.621130
2500,0.613823
3000,0.604361
3500,0.591947
4000,0.582445
4500,0.584235
5000,0.584176


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15750, training_loss=0.5842175089518229, metrics={'train_runtime': 1575.8968, 'train_samples_per_second': 159.888, 'train_steps_per_second': 9.994, 'total_flos': 1.053865593012015e+16, 'train_loss': 0.5842175089518229, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

ValueError: multi_class must be in ('ovo', 'ovr')

Labels:

0: Negative

1: Neutral

2: Positive

In [ ]:
from transformers import pipeline

# Load your saved model into a pipeline for easy testing
classifier = pipeline("text-classification", model="results")
result = classifier("what a bad day!")
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.893475353717804}]


In [ ]:
from huggingface_hub import login, upload_folder

# (optional) Login with your Hugging Face credentials
login()

# Push your model files
upload_folder(folder_path=".", repo_id="edaUsha/Fine_Tuning_Bert_For_Sentiment_Anaysis", repo_type="model")


It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-1000/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...point-12500/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  ...t-10000/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...nt-1000/model.safetensors:   0%|          | 14.2kB /  438MB            

  ...point-10500/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...t-10500/model.safetensors:   0%|          | 14.2kB /  438MB            

CommitInfo(commit_url='https://huggingface.co/edaUsha/Fine_Tuning_Bert_For_Sentiment_Anaysis/commit/e6ceb3189ef3c6b8e8d484604e6707851321b658', commit_message='Upload folder using huggingface_hub', commit_description='', oid='e6ceb3189ef3c6b8e8d484604e6707851321b658', pr_url=None, repo_url=RepoUrl('https://huggingface.co/edaUsha/Fine_Tuning_Bert_For_Sentiment_Anaysis', endpoint='https://huggingface.co', repo_type='model', repo_id='edaUsha/Fine_Tuning_Bert_For_Sentiment_Anaysis'), pr_revision=None, pr_num=None)